In [2]:
import sys
sys.path.append("..")  # so `feature_engineering.py` at the project root is importable

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
)
from xgboost import XGBClassifier

from feature_engineering import ChurnFeatureEngineer

In [3]:
df = pd.read_csv("../data/raw/Churn_Modelling.csv")
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
RAW_FEATURE_COLUMNS = [
    "CreditScore", "Geography", "Gender", "Age", "Tenure",
    "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
]

X = df[RAW_FEATURE_COLUMNS]
y = df["Exited"]


 Train/test split (test set stays untouched until final evaluation)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Build the pipeline

In [6]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Best params from the Optuna search in feateng_fixed_.ipynb (trial 8, CV ROC-AUC 0.8599)
best_xgb_params = {
    "max_depth": 3,
    "learning_rate": 0.02918496099190096,
    "n_estimators": 382,
    "min_child_weight": 10,
    "subsample": 0.7517629931052796,
    "colsample_bytree": 0.9351290346112869,
    "scale_pos_weight": scale_pos_weight,
    "eval_metric": "logloss",
    "random_state": 42,
}

pipeline = Pipeline([
    ("feature_engineering", ChurnFeatureEngineer()),
    ("model", XGBClassifier(**best_xgb_params)),
])

## Threshold tuning — leakage-free, on the raw training data

In [7]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_probs = cross_val_predict(
    pipeline, X_train, y_train,
    cv=cv_strategy,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_train, cv_probs)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_thresh = thresholds[f1_scores[:-1].argmax()]
print("Best threshold (from CV, no test-set leakage):", best_thresh)

Best threshold (from CV, no test-set leakage): 0.6219075


## Fit on the full training set, evaluate once on the held-out test set

In [8]:
pipeline.fit(X_train, y_train)

y_test_prob = pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_thresh).astype(int)

print(classification_report(y_test, y_test_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))
print("ROC AUC:", roc_auc_score(y_test, y_test_prob))

              precision    recall  f1-score   support

           0       0.91      0.90      0.90      1593
           1       0.61      0.65      0.63       407

    accuracy                           0.84      2000
   macro avg       0.76      0.77      0.77      2000
weighted avg       0.85      0.84      0.85      2000

Confusion matrix:
[[1427  166]
 [ 144  263]]
ROC AUC: 0.867266341842613


## Sanity check: pipeline works on a single raw record


In [9]:
sample = pd.DataFrame([{
    "CreditScore": 650,
    "Geography": "France",
    "Gender": "Female",
    "Age": 40,
    "Tenure": 3,
    "Balance": 60000.0,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 50000.0,
}])

proba = pipeline.predict_proba(sample)[:, 1][0]
print(f"Churn probability: {proba:.4f} -> {'churn' if proba >= best_thresh else 'stay'}")

Churn probability: 0.1752 -> stay


In [10]:
print(type(pipeline["model"]))

<class 'xgboost.sklearn.XGBClassifier'>


## Save the final deployable pipeline


In [ ]:
joblib.dump(
    {"pipeline": pipeline, "threshold": float(best_thresh)},
    "../models/churn_model_pipeline.pkl"
)

print("Saved pipeline to ../models/churn_model_pipeline.pkl")

Saved pipeline to ../models/churn_model_pipeline.pkl
